# Chapter 01 — AI fluency, prompting and output validation · Homework (self-test)

**Goal:** check that you can explain surprising output with the four model properties, pick the right mode and review level, write complete briefs, catch unsupported claims, compare drafts against a rubric, and turn recurring corrections into lasting fixes.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0** (everything runs offline; no API key needed)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 7 scenario questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises).

In [ ]:
import hashlib
import re
from collections import Counter
from typing import Callable

# ---------- grading bookkeeping (run, don't edit) ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")


NOTES = """Project Atlas weekly sync, 3 Nov 2026
- Migration: 18 of 24 tables moved. The last 6 wait for vendor API keys (owner: Priya, due 10 Nov).
- Budget: 41,200 EUR spent of 60,000 EUR.
- Decision: go-live moves from 14 Nov to 21 Nov."""

## Part A — Quiz (7 questions)

Pick one letter per question. Scenarios cover everyday business use, educators and students, in both Claude and ChatGPT.

**Q1. (Finance team)** An analyst pastes a 90-page supplier agreement and asks for every clause about price changes. The answer lists three clauses from the first half and misses an indexation clause in an annex near the end. Which model property best explains the miss?

- A) Knowledge: the model has not seen this supplier's contracts in training.
- B) Steerability: the question was phrased in a leading way.
- C) Working memory: attention over a very long input is uneven, so detail late in the document is easy to miss.
- D) Next-token prediction: the model invented the three clauses it did list.

**Q2. (Marketing)** A marketer asks a chat assistant, with no attachments, to "summarize what customers said about our product launch last month". The reply is a fluent summary with specific percentages. What is the right reaction?

- A) Treat the summary as unverified: the model cannot know last month's internal feedback, so attach the actual feedback (or use a connected source) and ask for a summary of that only.
- B) Accept it, because the percentages show the model did a careful analysis.
- C) Ask the model to rate its confidence and use the summary if the rating is high.
- D) Regenerate twice and use the figures that appear most often.

**Q3. (HR operations)** An HR lead wants to set up an assistant that answers staff questions about the leave policy, for everyone in the company to use. Which mode is this, and what does it need most?

- A) Automation; a one-line prompt is enough because the policy is clear.
- B) Augmentation; the HR lead should chat with each employee's question turn by turn.
- C) Automation; it needs only a quick read of the first answer.
- D) Agency; it needs carefully written instructions for how the assistant behaves, the policy as its source, test questions before launch, and monitoring and an escalation path after launch.

**Q4. (Project management)** A checker confirms that every number in an AI-written status update appears in the meeting notes, and that the update contains no quotes. The project manager is about to send it. What else must she check?

- A) Nothing: if every number is in the source, the update is accurate.
- B) Completeness and consistency: that every required point (decisions, owners, dates) is present and that no figure is used in the wrong context, such as an old go-live date presented as current.
- C) The reading level only, since the facts are already verified.
- D) Whether the model says it is confident in the update.

**Q5. (Sales operations)** A two-page prompt for drafting customer emails contains "never mention pricing" near the end, and earlier says "include all relevant commercial details". The drafts sometimes mention prices. What is the root cause and the fix?

- A) The model is not capable enough; switch to a bigger one.
- B) The prompt is too short; add more examples of good emails.
- C) Buried and conflicting instructions; resolve the conflict ("commercial details except pricing") and put the rules in one labelled block near the start.
- D) Steerability; add "IMPORTANT" in capitals to the pricing rule.

**Q6. (Educator)** A teacher asks ChatGPT for a ten-question quiz on a learning objective ("explain why the seasons change"). Several questions look good but two ask about the phases of the Moon, and one answer key repeats a common misconception. What review process fits?

- A) Check every question against the learning objective and the materials actually taught, check each answer key against a trusted source, then edit or replace what fails before students see it.
- B) Ask ChatGPT whether the quiz matches the objective and use it if it says yes.
- C) Use the quiz as is, because most questions are fine.
- D) Regenerate the quiz until no question mentions the Moon.

**Q7. (Student)** A student uses an AI assistant to build an outline for an essay. Her university allows AI for planning if it is disclosed, and the essay must be based on the course readings. What should she do?

- A) Use the outline and its suggested sources without mentioning the AI, since it only helped with planning.
- B) Ask the AI to write the essay too, because the outline was allowed.
- C) Avoid AI entirely, since any use is risky.
- D) Check each point and source in the outline against the course readings, drop what is not supported, write the essay herself, and disclose the AI use as the policy requires.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': 'ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Find the gaps in a brief

Write `brief_gaps(brief, task_type)`. `brief` is a dict of component name → text. Return a list:

1. every name in `REQUIRED` whose value is missing or only whitespace, **in `REQUIRED` order**;
2. then, if the task type's extra need (`TYPE_NEEDS[task_type]`: `criteria` for analysis, `sources` for research, `audience` for drafting, `count` for brainstorming) is missing or blank, append that name.

Raise `ValueError` for an unknown `task_type`. Theory: [1.4](README.md#14-description-a-repeatable-prompt-structure), [1.6](README.md#16-matching-the-prompt-to-the-task-type)

In [ ]:
COMPONENTS = ("task", "context", "inputs", "output", "constraints", "examples", "when_unsure")
REQUIRED = ("task", "context", "inputs", "output", "when_unsure")
TYPE_NEEDS = {"analysis": "criteria", "research": "sources", "drafting": "audience",
              "brainstorming": "count"}


def brief_gaps(brief: dict[str, str], task_type: str) -> list[str]:
    """Missing required components (in REQUIRED order), then the task type's extra need."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        full = {"task": "Draft the launch email.", "context": "For existing customers.",
                "inputs": "Release notes v2.3", "output": "Under 120 words.",
                "when_unsure": "Say 'not stated'.", "audience": "Non-technical customers"}
        assert brief_gaps(full, "drafting") == [], f"complete drafting brief: got {brief_gaps(full, 'drafting')}"
        got = brief_gaps({"task": "Summarize", "context": "  \n", "output": "5 bullets"}, "analysis")
        assert got == ["context", "inputs", "when_unsure", "criteria"], (
            f"expected ['context', 'inputs', 'when_unsure', 'criteria'] (whitespace counts as missing, "
            f"REQUIRED order, extra need last), got {got}")
        got = brief_gaps(dict(full, count="20 options"), "brainstorming")
        assert got == [], f"brainstorming brief with count: got {got}"
        assert brief_gaps(full, "research") == ["sources"], "a research brief needs 'sources'"
        try:
            brief_gaps(full, "poetry")
        except ValueError:
            pass
        else:
            raise AssertionError("an unknown task_type must raise ValueError")
    _run_check(1, check)


check_ex1()

### Exercise 2 — Flag unsupported numbers and quotes

Write `unsupported(answer, source)` returning `{"numbers": [...], "quotes": [...]}`:

- **numbers:** every number in `answer` that is not in `source`, as a **sorted** list of strings with commas removed (`"41,200"` and `"41200"` are the same number). Use the pattern `r"\d[\d,.]*\d|\d"` so `"10."` at the end of a sentence reads as `"10"`.
- **quotes:** every span inside double quotes in `answer` that does not appear in `source`, comparing **case-insensitively** and with **runs of whitespace collapsed to one space**. Keep them in the order they appear.

Theory: [1.7](README.md#17-discernment-accuracy-completeness-consistency-and-bias)

In [ ]:
def unsupported(answer: str, source: str) -> dict[str, list[str]]:
    """{'numbers': sorted numbers not in source, 'quotes': quoted spans not in source}."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        ok = "Budget: 41200 EUR spent of 60,000 EUR; go-live moves to 21 Nov."
        assert unsupported(ok, NOTES) == {"numbers": [], "quotes": []}, (
            f"commas must not matter: got {unsupported(ok, NOTES)}")
        bad = ('About 90% done, 45,000 EUR spent, 2.5 weeks late. The team said "no blockers" '
               'and "The last   6 wait for vendor API keys".')
        got = unsupported(bad, NOTES)
        assert got["numbers"] == ["2.5", "45000", "90"], f"numbers: expected ['2.5', '45000', '90'], got {got['numbers']}"
        assert got["quotes"] == ["no blockers"], (
            f"quotes: expected ['no blockers'] (the second quote is in the notes once whitespace "
            f"and case are normalized), got {got['quotes']}")
        assert unsupported("Keys due 10.", NOTES)["numbers"] == [], "'10.' at a sentence end is the number 10"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Choose the review level

Write `review_level(task)`. `task` is a dict with optional keys `stakes` (`"low"`, `"medium"`, `"high"`; default low), `external`, `reversible` (default `True`), `checkable` (default `True`), `regulated`, `novel` (booleans, default `False` unless stated). Apply in order:

1. `"expert_signoff"` if stakes are high, **or** it is regulated (legal, medical, financial advice), **or** you cannot check it yourself;
2. `"fact_check"` if it goes outside the team, **or** cannot be undone, **or** stakes are medium, **or** it is a new kind of task;
3. otherwise `"self_check"`.

Theory: [1.8](README.md#18-when-a-human-must-verify)

In [ ]:
def review_level(task: dict) -> str:
    """'expert_signoff' | 'fact_check' | 'self_check' (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        cases = [
            ({}, "self_check"),
            ({"stakes": "low", "external": True}, "fact_check"),
            ({"reversible": False}, "fact_check"),
            ({"stakes": "medium"}, "fact_check"),
            ({"novel": True}, "fact_check"),
            ({"stakes": "high", "external": False}, "expert_signoff"),
            ({"regulated": True, "stakes": "low"}, "expert_signoff"),
            ({"checkable": False}, "expert_signoff"),
            ({"stakes": "low", "external": False, "reversible": True, "checkable": True}, "self_check"),
        ]
        for task, want in cases:
            got = review_level(task)
            assert got == want, f"review_level({task}) = {got!r}, expected {want!r}"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Compare drafts with a rubric

Write `rubric_score(draft, must_include, max_words, banned)`: **+1** for each `must_include` term present, **+1** if the draft has at most `max_words` words (split on whitespace), **−1** for each `banned` term present (all matching case-insensitive, as substrings).

Write `pick_draft(drafts, must_include, max_words, banned)`: return the id of the draft with the **highest** score; break ties by **fewer words**, then by **alphabetical id**. Theory: [1.9](README.md#19-editing-comparing-and-formatting-for-the-audience)

In [ ]:
def rubric_score(draft: str, must_include: list[str], max_words: int, banned: list[str]) -> int:
    """+1 per must_include term present, +1 if within max_words, -1 per banned term present."""
    # TODO
    raise NotImplementedError


def pick_draft(drafts: dict[str, str], must_include: list[str], max_words: int,
               banned: list[str]) -> str:
    """Id of the best draft: highest score, then fewest words, then alphabetical id."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        rub = dict(must_include=["21 Nov", "41,200"], max_words=12, banned=["synerg"])
        assert rubric_score("Go-live 21 nov; 41,200 EUR spent.", **rub) == 3, "both terms (case-insensitive) + short"
        assert rubric_score("Synergies! Go-live 21 Nov.", **rub) == 1, "one term + short - one banned = 1"
        assert rubric_score(" ".join(["word"] * 13), **rub) == 0, "13 words is over max_words=12"
        drafts = {
            "warm": "Great synergies across the team, launch soon!",
            "b_complete": "Go-live 21 Nov. Spent 41,200 EUR of 60,000 EUR so far.",
            "a_complete": "Go-live 21 Nov. Spent 41,200 EUR of 60,000 EUR to date.",
            "short": "Go-live 21 Nov; 41,200 EUR spent.",
        }
        assert pick_draft(drafts, **rub) == "short", "highest score, then fewest words"
        del drafts["short"]
        assert pick_draft(drafts, **rub) == "a_complete", "same score and length: alphabetical id wins"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Make recurring corrections persist

Normalize a correction by collapsing runs of whitespace to one space, stripping, and lower-casing.

- `promote(corrections, threshold=3)`: return the normalized corrections that occur at least `threshold` times, **most frequent first**; on a tie, the one seen first comes first.
- `update_instructions(instructions, promoted)`: return `instructions` (unchanged, same order) followed by each promoted correction that is **not already there** (compare normalized).

Theory: [1.10](README.md#110-the-description-discernment-loop-diagnosing-weak-output)

In [ ]:
def promote(corrections: list[str], threshold: int = 3) -> list[str]:
    """Normalized corrections seen at least `threshold` times, most frequent first."""
    # TODO
    raise NotImplementedError


def update_instructions(instructions: list[str], promoted: list[str]) -> list[str]:
    """instructions + promoted corrections not already present (compared normalized)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        log = ["Use fiscal quarters", "list owners  by name", "Bold blocked items", "use fiscal quarters ",
               "List owners by name", "USE FISCAL QUARTERS", "list owners by name", "bold blocked items"]
        got = promote(log)
        assert got == ["use fiscal quarters", "list owners by name"], (
            f"expected ['use fiscal quarters', 'list owners by name'] (normalized, tie -> first seen), got {got}")
        assert promote(log, threshold=2) == ["use fiscal quarters", "list owners by name", "bold blocked items"]
        assert promote([]) == [], "an empty log promotes nothing"
        base = ["Five bullets, plain English.", "Use Fiscal Quarters"]
        new = update_instructions(base, got)
        assert new == ["Five bullets, plain English.", "Use Fiscal Quarters", "list owners by name"], (
            f"keep existing instructions, add only new ones: got {new}")
        assert base == ["Five bullets, plain English.", "Use Fiscal Quarters"], "do not mutate the input list"
    _run_check(5, check)


check_ex5()

## Part C — Architecture scenario

**AI fluency for a customer-success team.** A software company has a 40-person customer-success team that has just been given both Claude and ChatGPT enterprise seats. Each person writes about 30 customer emails, 5 account summaries and 1 quarterly business review (QBR) deck a week. Six weeks in, the team lead reports three problems: account summaries sometimes contain renewal figures that are not in the CRM export; emails are long and full of internal product names; and everyone keeps correcting the same things in every chat. One email quoted a discount that had never been approved. Leadership asks you for a one-page plan.

Describe your plan (15–25 lines):
1. Which tasks are automation, augmentation or agency, and what the team keeps.
2. The standard brief for the account summary and for customer emails (components, sources, "when unsure").
3. Where each kind of context lives in Claude and in ChatGPT, so it persists.
4. How outputs are checked: which checks, and which review level for summaries, emails, QBRs and any pricing.
5. How the team diagnoses weak output and turns recurring corrections into fixes.
6. Diligence: data rules, disclosure, ownership, and how you will know the plan is working.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Modes:** emails and account summaries are **automation** from a standard brief; QBR decks are **augmentation** (the CSM owns the story and the judgment); a shared Project or custom GPT with the team's instructions is **agency** and needs testing before rollout. Sending, pricing and commitments stay with humans.
- **Briefs:** task, context (customer, relationship, goal), inputs (the CRM export or ticket history, clearly marked), output (length, structure, tone for the customer), constraints ("only figures from the export", "never state discounts or prices"), one approved example, and "write 'not in the export'" instead of guessing.
- **Context placement:** per-customer data in the chat; team rules, approved examples and the product-name glossary in **Project instructions and knowledge** (Claude) and **project instructions and files** or a **custom GPT** (ChatGPT); personal style in profile preferences / custom instructions. One owner keeps them current.
- **Discernment:** every figure traced to the CRM export (a simple unsupported-number check); required points present (renewal date, owner, next step); jargon check against the glossary; swap test or spot check for tone differences between customers. Not "ask the model if it is sure".
- **Review levels:** internal summaries **fact-check**; customer emails **fact-check** by the sender; anything with pricing, discounts or contract terms **expert sign-off** (deal desk), because it is external, high-stakes and hard to undo. The unapproved discount is an incident: trace it to the missing constraint and the missing review step.
- **Diagnosis loop:** symptom → root cause → one change (invented renewals = missing source/"only" rule; long jargon = no audience/output; repeated fixes = nothing persistent). A shared correction log; anything corrected three times goes into the Project instructions or template, then re-tested on a few real cases.
- **Diligence:** only enterprise seats for customer data; no personal accounts; disclosure where customers or policy expect it; the sender owns every email. **Measure:** weekly sample of outputs scored against the rubric, count of corrections per output, unsupported figures found, time per email, and zero unapproved commitments.
- **Both vendors:** the same brief, checks and review levels work in Claude and ChatGPT; only feature names differ. Strong answers note that feature names and admin controls change and should be checked in each vendor's help pages.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_q, n_ex = 7, 5
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (n_q + n_ex)
print(f"Quiz:       {quiz}/{n_q} ({quiz / n_q:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 02" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next: [Chapter 02 — Workflow design and delegation](../02-workflow-design-and-delegation/README.md). Back to the [track overview](../README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** The clauses were in the input, so this is not a knowledge gap (A). The question was neutral (B). Nothing suggests the three listed clauses are invented (D). The fix is to point at the annex or split the document, and to check completeness against the table of contents.

**Q2:** **A.** Recent internal information is outside the model's knowledge, so specific figures are next-token guesses. B mistakes specificity for evidence. C relies on a self-rating, which is not evidence. D can make guesses agree without making them true.

**Q3:** **D.** Configuring AI to act later for other people is agency. It needs performance description (instructions), testing, and deployment diligence. A and C treat a shared, ongoing assistant as a single task. B does not scale and is not what the HR lead is building.

**Q4:** **B.** A number can appear in the source and still be wrong in context (the old date), and a check of what is present says nothing about what is missing. A overstates what the number check proves. C skips completeness. D relies on a self-rating.

**Q5:** **C.** The prompt asks for two incompatible things and hides one of them. A and B do not remove the conflict. D makes one rule louder but leaves the contradiction in place.

**Q6:** **A.** The sources of truth are the objective, the taught materials and a trusted reference; students are the audience, so errors have real cost. B is self-assessment. C ships known errors. D removes one symptom and never checks the answer keys.

**Q7:** **D.** That is discernment (check against her own materials) plus diligence (follow the rules, disclose, own the work). A breaks the disclosure rule and trusts unverified sources. B goes beyond what the policy allows. C ignores a use that is explicitly allowed.
</details>

<details><summary>Exercise 1 — Find the gaps in a brief: reference solution</summary>

```python
COMPONENTS = ("task", "context", "inputs", "output", "constraints", "examples", "when_unsure")
REQUIRED = ("task", "context", "inputs", "output", "when_unsure")
TYPE_NEEDS = {"analysis": "criteria", "research": "sources", "drafting": "audience",
              "brainstorming": "count"}


def brief_gaps(brief: dict[str, str], task_type: str) -> list[str]:
    if task_type not in TYPE_NEEDS:
        raise ValueError(f"unknown task type: {task_type}")
    blank = lambda key: not str(brief.get(key, "")).strip()
    missing = [c for c in REQUIRED if blank(c)]
    if blank(TYPE_NEEDS[task_type]):
        missing.append(TYPE_NEEDS[task_type])
    return missing
```
</details>

<details><summary>Exercise 2 — Flag unsupported numbers and quotes: reference solution</summary>

```python
NUM = re.compile(r"\d[\d,.]*\d|\d")


def unsupported(answer: str, source: str) -> dict[str, list[str]]:
    norm = lambda s: " ".join(s.split()).lower()
    nums = lambda s: {n.replace(",", "") for n in NUM.findall(s)}
    quotes = [q for q in re.findall(r'"([^"]+)"', answer) if norm(q) not in norm(source)]
    return {"numbers": sorted(nums(answer) - nums(source)), "quotes": quotes}
```
</details>

<details><summary>Exercise 3 — Choose the review level: reference solution</summary>

```python
def review_level(task: dict) -> str:
    if task.get("stakes") == "high" or task.get("regulated") or not task.get("checkable", True):
        return "expert_signoff"
    if (task.get("external") or not task.get("reversible", True)
            or task.get("stakes") == "medium" or task.get("novel")):
        return "fact_check"
    return "self_check"
```
</details>

<details><summary>Exercise 4 — Compare drafts with a rubric: reference solution</summary>

```python
def rubric_score(draft: str, must_include: list[str], max_words: int, banned: list[str]) -> int:
    text = draft.lower()
    points = sum(term.lower() in text for term in must_include)
    points += len(draft.split()) <= max_words
    points -= sum(word.lower() in text for word in banned)
    return points


def pick_draft(drafts: dict[str, str], must_include: list[str], max_words: int,
               banned: list[str]) -> str:
    return min(drafts, key=lambda k: (-rubric_score(drafts[k], must_include, max_words, banned),
                                      len(drafts[k].split()), k))
```
</details>

<details><summary>Exercise 5 — Make recurring corrections persist: reference solution</summary>

```python
def normalize(correction: str) -> str:
    return " ".join(correction.split()).lower()


def promote(corrections: list[str], threshold: int = 3) -> list[str]:
    counts = Counter(normalize(c) for c in corrections)      # Counter keeps first-seen order on ties
    return [c for c, n in sorted(counts.items(), key=lambda kv: -kv[1]) if n >= threshold]


def update_instructions(instructions: list[str], promoted: list[str]) -> list[str]:
    have = {normalize(i) for i in instructions}
    return instructions + [p for p in promoted if normalize(p) not in have]
```
</details>